# SpurLab × spur-graph: user and agent experience

Research notebook • 2026-11-16 • tracking: `bd-1l47`

**Question:** Starting with a task, can a person and their agent discover the right DuckDB capability, run it in the right environment, get a useful result, understand its limits, and resume safely?

This investigation steps back from the product website. It examines **SpurLab as the notebook/agent host**, its DuckDB extension workflow, and **`spur_graph` as the native SQL package**. No separate package named `spur_lab` has been established by the evidence. Naming remains a product-contract question.

Evidence labels: **Observed** = live tool/query result; **Source** = current implementation/documentation inspected; **Proposed** = target experience; **Unknown** = not exercised here. A source capability or registered function alone does not establish a successful user journey.

The notebook will keep the managed SQL connection and any native DuckDB 2.0.0 experiment explicitly separate. Existing user notebooks and source edits are preserved.

## Decision to carry forward

**Build one complete, honest task loop around the existing kernel.** The next product milestone should let a user say: “Use this repository snapshot to show callers and unresolved calls, with source locations; save the result so I can inspect and rerun it.”

Offer two explicit capability levels:

- **Single-file facts:** supported input is a relative logical path plus source text. Output is typed syntax-derived facts, local resolution, evidence, and uncertainty. This already has a working native SQL seam.
- **Repository analysis:** requires source admission, snapshot identity, orchestration, cross-file resolution policy, durable materialization, and query recipes. The extension alone does not establish these capabilities.

SpurLab is the task workspace; DuckDB is the query/runtime boundary; `spur_graph` supplies extraction and pair-matching primitives; caller workflows supply repository identity and orchestration. Keep this boundary visible in the capability contract.

The most urgent gaps are **getting onto a supported runtime**, **finding and learning the package through the normal UI/tools**, and **preserving evidence correctly when turning facts into an answer**. The existing cache and typed Parquet workflow are useful foundations.

### What is verified here

Native DuckDB 2.0.0 ran the hash-checked local package. Managed SQL stayed on 1.5.5 and successfully queried the typed exports using full input keys. A fresh native connection read cached data but had no graph functions loaded. These are three different readiness statements.

Human interaction findings below come from current UI source. They have not been validated through a novice usability session or a live screenshot walkthrough.

In [2]:
SELECT version() AS duckdb_version;

SELECT extension_name, installed, loaded, install_mode
FROM duckdb_extensions()
WHERE installed OR loaded
ORDER BY extension_name;

SELECT function_name, function_type, parameters, parameter_types, return_type,
       description, examples
FROM duckdb_functions()
WHERE starts_with(function_name, 'spur_')
ORDER BY function_name;

function_name,function_type,parameters,parameter_types,return_type,description,examples
spur_native_refresh,table_macro,"[""source_name""]",[null],NULL,NULL,[]
spur_rest_table,table,"[""col0"", ""col1""]","[""VARCHAR"", ""VARCHAR""]",NULL,NULL,[]


In [3]:
SELECT version() AS duckdb_version,
       current_setting('allow_unsigned_extensions') AS allow_unsigned_extensions,
       current_database() AS current_database;


duckdb_version,allow_unsigned_extensions,current_database
v1.5.5,True,memory


In [4]:
SELECT extension_name, installed, loaded, install_mode
FROM duckdb_extensions()
WHERE installed OR loaded
ORDER BY extension_name;

extension_name,installed,loaded,install_mode
a5,True,False,REPOSITORY
adbc,True,False,REPOSITORY
adbc_scanner,True,False,REPOSITORY
agent_data,True,False,REPOSITORY
airport,True,False,REPOSITORY
altertable,True,False,REPOSITORY
anndata,True,False,REPOSITORY
anofox_forecast,True,False,REPOSITORY
anofox_optimize,True,False,REPOSITORY
anofox_scenario,True,False,REPOSITORY


In [5]:
import sys, subprocess
from pathlib import Path

native_python = Path('/private/tmp/spur-graph-duckdb-v2-20261114/bin/python')
assert native_python.exists(), 'Pinned native DuckDB environment is missing'
probe = subprocess.run([str(native_python), '-c', 'import duckdb, sys; print(sys.executable); print(duckdb.__version__)'], capture_output=True, text=True, timeout=20)
print('Notebook Python:', sys.executable)
print('Separate native process:')
print(probe.stdout)
if probe.stderr: print(probe.stderr)
assert probe.returncode == 0


Notebook Python: /Users/kevintruong/.spur/jupyter/venv/bin/python
Separate native process:
/private/tmp/spur-graph-duckdb-v2-20261114/bin/python
2.0.0



In [7]:
import subprocess
from pathlib import Path

# Explicit boundary: this process uses native DuckDB 2.0.0; SQL cells still use managed 1.5.5.
probe_script = Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-16-extension-agent-experience/native_probe.py')
run = subprocess.run([str(native_python), str(probe_script)], capture_output=True, text=True, timeout=60)
print(run.stdout)
if run.stderr: print(run.stderr)
assert run.returncode == 0, f'Native experience probe failed: {run.returncode}'


{
  "observed_at_utc": "2026-11-16T05:56:04.877426+00:00",
  "runtime": {
    "python": "/private/tmp/spur-graph-duckdb-v2-20261114/bin/python",
    "duckdb": "v2.0.0",
    "scope": "separate native Python process; not managed notebook SQL"
  },
  "artifact": {
    "path": "/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-15-duckdb-file-facts/integration/spur_graph.duckdb_extension",
    "sha256": "e7d191bcda2e0e8ea52d249189007e1a7e424f1b2b8fb987dfebde037a575028",
    "build_commit_from_prior_protocol": "f0218c673ed478f854b64df6223d1817ced3586e"
  },
  "source_head_at_observation": "5aab2625bd19f2fd0da7e37cbf352ad39c8425aa",
  "first_result": {
    "symbol_names": [
      "helper",
      "entry"
    ]
  },
  "single_file": {
    "list_counts": {
      "nodes": 3,
      "edges": 4,
      "spans": 3,
      "macro_invocations": 0,
      "macro_scopes": 0
    },
    "edge_relations": {
      "contains": 2,
      "calls": 2
    },
    "call_evidence": [
      {
        "s

## What the probes establish

**The native kernel already works; the path from a user task to a trustworthy answer is incomplete.** On this machine a fresh managed notebook selects **DuckDB 1.5.5**, while the prototype package is pinned to **2.0.0**. The separate native probe loaded a hash-checked existing artifact and ran successfully. It did not install the extension into this notebook's managed SQL connection.

| Evidence | Observed result | Experience implication |
|---|---|---|
| Refreshed `notebook_search_extensions(query="spur_graph")` | Zero cards for the selected 1.5.5 macOS arm64 target | The normal discovery path did not offer this capability here; global publication status is unproven |
| Managed runtime function search | Zero `spur_graph` matches, before and after running native Python | Another process's successful load does not make notebook SQL ready |
| Reference function search for graph | Zero cards in a partial reference catalog | An agent cannot obtain graph-specific guidance from this route today |
| Native 2.0.0 function registration | Three functions, all with descriptions; example counts **1, 0, 0** | A useful primitive API exists; runnable task recipes are thin |
| One-file call fixture | `helper` resolves locally; `remote_call` has NULL target and `syntax_exact` confidence | Exact syntax evidence must not be presented as resolved program semantics |
| Two file inputs | Numeric node IDs 1, 2, 3 repeat | Full input identity must accompany IDs in joins and citations |
| Exact cache workflow, two runs | Missing keys **5 → 0**, child refresh keys **0** on replay; **5** parents, **10** lineage occurrences | Durable, reusable facts already work in this small fixture |
| Invalid/unsupported paths | Explicit errors; connection remains usable; `TRY` returns NULL | Basic engine recovery works; cached error classification loses the original message |
| New native connection without LOAD | Zero graph functions, five cached parent rows remain readable | Durable data and connection readiness are separate states |

The artifact comes from a prior build, identified by SHA-256 in `native-results.json`; these checks do not certify that every current source change is in that binary. The source and artifact identities are recorded separately. Synthetic fixture results establish behavior on these inputs, not general extraction accuracy or performance.

The first native probe failed because this investigation used the removed DuckDB 2.0 relation `.fetchone()` API. Switching the harness to `connection.execute(...).fetchone()` corrected it. This was an experiment-authoring error, not an extension failure.

In [8]:
-- Read typed native exports on the managed DuckDB 1.5.5 connection.
-- This demonstrates data interoperability, without loading spur_graph here.
WITH nodes AS (
  SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-16-extension-agent-experience/file-facts-nodes.parquet')
), edges AS (
  SELECT * FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-16-extension-agent-experience/file-facts-edges.parquet')
)
SELECT e.logical_path, n.label AS caller, e.target_label AS callee,
       e.target_node_id IS NOT NULL AS target_resolved_in_file,
       e.confidence, e.bind_method
FROM edges e
LEFT JOIN nodes n
  ON e.blob_oid = n.blob_oid AND e.logical_path = n.logical_path
 AND e.extractor_sha = n.extractor_sha AND e.source_node_id = n.node_id
WHERE e.relation = 'calls'
ORDER BY e.logical_path, e.edge_id;

logical_path,caller,callee,target_resolved_in_file,confidence,bind_method
src/copy.rs,entry,helper,True,syntax_exact,singleton
src/copy.rs,entry,remote_call,False,syntax_exact,NULL
src/demo.rs,entry,helper,True,syntax_exact,singleton
src/demo.rs,entry,remote_call,False,syntax_exact,NULL


In [9]:
SELECT extraction_status, count(*) AS lineage_occurrences
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-16-extension-agent-experience/file-facts-occurrences.parquet')
GROUP BY extraction_status ORDER BY extraction_status;

extraction_status,lineage_occurrences
empty_source,2
extraction_error,2
null_input,2
ok,4


## The human journey: decisions and friction

| Stage | What the user should experience | Current evidence | Work needed |
|---|---|---|---|
| Express intent | “Find callers in this repo snapshot” selects a relevant recipe | The kernel takes path + source, not a repository URL or directory | Name the supported task and input scope before offering setup |
| Find capability | Search “code symbols”, “callers”, or “repository changes” and see capabilities and limits | Source has an Extensions sidebar, catalog, target label, setup and Functions / Capabilities sections; selected-target graph search returned zero cards | Add graph package metadata and task vocabulary; explain missing/incompatible results |
| Choose runtime | See a supported version/platform and an explicit way to prepare it | Current managed target is 1.5.5; package is exactly 2.0.0 | Ship one coherent supported runtime/artifact path; preserve the selected target across UI, agent tools and execution |
| Prepare | A notebook-owned setup with clear progress and restart behavior | Generic catalog setup already saves setup, checks target changes and offers retry | Integrate graph with this lifecycle; a manual Python subprocess is a research route, not a finished setup experience |
| Supply data | Select a frozen repository or staged source relation; preview scope and exclusions | Extension deliberately does no filesystem or Git discovery; examples provide separate ingestion helpers | A source-admission recipe should state revision, dirty-file policy, languages, exclusions, limits and credentials if needed |
| First useful result | See a small table of symbols/calls, source evidence, and unresolved targets | Native fixture and portable managed SQL query succeeded | Provide runnable samples and explain each evidence field |
| Inspect and refine | Follow a row to the file/range; narrow the task without losing provenance | Facts contain spans and local IDs; our probe shows repeated numeric IDs across files | Join/citation helpers should always retain full keys and state how source bytes match the answer |
| Resume and share | Reopen the notebook and know which data are reusable and which runtime steps must run | Cached data survives a fresh connection; native functions require LOAD again | Persist data/artifact/recipe identities; refresh readiness before reuse; expose migration/rebuild choices |

Current UI foundations should be extended: [ExtensionsPanel.tsx](/Users/kevintruong/project/spur-notebook/jute-notebook/src/ui/notebook/sidebar/extensions/ExtensionsPanel.tsx), [DuckDbCatalog.tsx](/Users/kevintruong/project/spur-notebook/jute-notebook/src/ui/notebook/sidebar/extensions/DuckDbCatalog.tsx), [DuckDbSetup.tsx](/Users/kevintruong/project/spur-notebook/jute-notebook/src/ui/notebook/sidebar/extensions/DuckDbSetup.tsx). Curated extension identities currently do not include a graph entry: [types.rs](/Users/kevintruong/project/spur-notebook/src/extensions/types.rs:11). That enum is a curated surface, not an exhaustive claim about every catalog package.

## The agent journey: facts it must obtain

| Agent decision | Existing route | Current graph gap / required contract |
|---|---|---|
| Which notebook and connection? | `notebook_context_pack` → selected target + `next_queries` | Keep notebook, environment, engine/ABI, platform and runtime generation explicit |
| Is there a suitable package? | `notebook_search_extensions`, then exact returned card/snapshot/target | Graph yielded no card here. Return actionable reasons and alternatives; a blank list is insufficient guidance for this task |
| Can I prepare it safely and retry? | `notebook_add_extension` with exact discovery values and mutation UUID; inspect interrupted operation with `notebook_list_extensions` | Integrate graph into the existing saved-setup contract, including target mismatch, unavailable build and interrupted reply cases |
| Is it callable now? | Refresh `notebook_list_extensions`; search `notebook_duckdb_functions(source="runtime")` | Require current-connection load/signature evidence. Installed files, saved membership and another process's success do not establish this |
| How do I invoke it? | Function search → returned `duckdbfn://` ref → `notebook_duckdb_function_get` | Supply full signatures, small runnable recipes, supported languages, NULL/empty/error semantics, cost and scope limits. Two native functions have no registered examples |
| Where do inputs come from? | Notebook catalog/schema navigation, then staged relations | A frozen Git/source adapter and explicit lineage contract are needed for repository tasks; avoid inventing columns or assuming the extension reads files |
| Can I compose the result? | Typed SQL structs/lists and independent child tables | Teach full-key joins, separate list expansion, local-ID scope and retention of parents with zero children |
| Can I trust this answer? | Current rows plus source/lineage/evidence fields | Distinguish local resolution, unresolved calls, extraction failure, unsupported input, stale data and temporal ambiguity |
| Can I inspect the complete result? | `notebook_run_cell` / `notebook_read_cell` currently provide SQL table text; bounded chunk reads exist | The inventory displayed “50 rows (20 shown)”. Chunking saved text cannot restore omitted rows. Use bounded aggregate queries today; add a typed, paginated result route for exact agent consumption |
| Can I resume? | Saved notebook, data cache, operation receipts and refreshed runtime observations | Reopen/restart must re-establish readiness and compare source/artifact/recipe identity before executing or citing cached results |

Existing tools already distinguish passive discovery from setup and execution. Preserve that separation. Unknown function ownership in a runtime catalog must stay unknown; prefix matching can select names but cannot prove package ownership.

A minimal successful agent response should identify the **task, data snapshot, runtime/artifact, query, rows and coverage, uncertainty, evidence references, persisted output, and next recovery action if incomplete**. A small typed result manifest should carry these fields; its name and schema are proposed, not existing APIs.

## What the native API does—and where orchestration begins

| API / workflow | Current contract | Agent limitation |
|---|---|---|
| `spur_graph_symbols(logical_path, source)` | Typed symbol list, names/kinds/scopes, byte and line ranges, anchors/tokens | No repository discovery, call graph, embeddings or history walk |
| `spur_graph_file_facts(logical_path, source)` | Nodes, edges, spans, macro invocations and scopes; one-file native extraction and pending-reference resolution | Cross-file targets may remain unresolved; numeric IDs are local to each input |
| `spur_graph_symbol_changes(path, old_path, kind, old_symbols, new_symbols)` | Typed changes + diagnostics for one supplied version pair | Git inventory, commit/parent lineage and source staging remain caller work |
| `examples/file_facts.sql` | Durable full-key cache, five typed children, occurrence lineage, checkpoint repair and seven typed exports | Caller supplies correct immutable identities; errors are cached without text; exports are replaced as a non-atomic set |
| Repository/history examples | Frozen Git catalogs, staged extraction and pair-matching workflows | Experimental orchestration and admission limits need a supported task wrapper |

**Key rules for an agent:**

- File-facts identity is `(blob_oid, logical_path, extractor_sha)`; retain it when joining a local `node_id` or `edge_id`. Do not substitute the temporal symbol cache's `(blob_oid, language)` key.
- Empty source, empty facts, NULL input and extraction error are different outcomes. A parent with no child rows may still be complete; keep the parent/status relation.
- Expand each nested list separately. Joining independent lists together can multiply rows.
- A NULL target does not establish “no dependency.” Our `remote_call` edge has exact syntactic evidence and no resolved target.
- An ambiguous rename should remain a diagnostic. Our old/new-name fixture produced deletion + addition and `ambiguous_rename`, not a confident rename.
- Cache replay skipped extraction for all five fixture keys. Failure statuses also remained cached. Retrying a failed key requires explicit invalidation; changing the extractor hash creates new keys.
- Input-byte budgets do not bound parser AST, nested output, or global temporal-finalization memory. Do not promise bounded repository processing from batch size alone.

Source: [README and SQL contract](../../crates/spur-graph-duckdb/README.md), [entrypoint and registration](../../crates/spur-graph-duckdb/src/lib.rs), [file-facts adapter](../../crates/spur-graph-duckdb/src/file_facts.rs), [cache workflow](../../crates/spur-graph-duckdb/examples/file_facts.sql). The current-worktree graph was attempted first; stale/limited retrieval required checking raw working-tree bytes. Source hashes and runtime artifact hashes are recorded separately.

## Prioritized work: a complete first task

These are proposed implementation work items, not a claim that new implementation has been authorized or dispatched. Priority follows blocking dependencies and correctness risk; it is not a numerical score.

| ID | Priority | Concrete change | Primary boundary | Acceptance evidence | Depends on |
|---|---|---|---|---|---|
| W1 | P0 | Decide and publish a supported runtime/ABI/platform path for `spur_graph`; wire it to the selected notebook target | SpurLab runtime + extension packaging | A fresh notebook completes supported preparation; incompatible targets explain the required version without attempting a wrong load | — |
| W2 | P0 | Add a discoverable graph capability card, scope/limitations, package ownership evidence where available, and runnable examples for all three functions | Package documentation + host catalog | User and agent can discover by task terms and obtain exact signatures/examples through the standard routes | W1 support matrix |
| W3 | P0 | Ship a notebook recipe for “source → symbols/calls → checked result”, with a built-in small fixture and an optional staged-source input | Notebook recipe + source admission | First useful result requires no hand-written artifact paths or external helper discovery; it displays unresolved targets and provenance | W1, W2 |
| W4 | P0 | Define trustworthy result semantics: full identity keys, resolution status, coverage, parent statuses, temporal diagnostics and stale-source checks | Extraction/query contract | Two-file ID-collision and unresolved-call fixtures cannot produce a confidently wrong cross-file join or “no callers” answer | W3 |
| W5 | P1 | Wrap frozen repository ingestion in a resumable operation with supported languages, exclusions, bounds, progress, cancellation, failure reasons and explicit retry policy | Repository orchestration | Known multi-file fixture preserves all admitted/failed/skipped parents; resume reuses compatible keys; scope and unresolved coverage are visible | W3, W4 |
| W6 | P1 | Add typed bounded result retrieval and source evidence navigation; retain readable notebook tables | Notebook agent output + UI | Agent retrieves exact typed rows/schema and completeness even when table display is abbreviated; human opens matching file/range | W4 |
| W7 | P1 | Connect graph setup and saved recipes to existing retry/restart/target-change lifecycle | Host extension lifecycle | Restart revalidates readiness; reply-loss retry preserves operation identity; incompatible extractor/schema changes give an explicit rebuild/migration path | W1, W3 |
| W8 | P2 | Promote history/finalization and production admission policies, then expand distribution/platform coverage | History pipeline + packaging | Equivalent-workload correctness, restart receipts, resource limits and publication compatibility are demonstrated per supported target | W5, W7 |

**Suggested implementation order:** W1 and W2 establish the contract; W3 delivers one small end-to-end task; W4 fixes the interpretation contract before expanding scope; then W5–W7 complete repository use, exact result access and lifecycle recovery. W8 follows those gates. Runtime choice and truthful evidence deserve attention before broad catalog growth or website work.

**Design decisions still needed:** Is the first supported promise single-file facts or a fully prepared repository graph? Which runtime/distribution is supported first? Are dirty working-tree inputs part of that promise? How should a user choose between cached syntax evidence and deeper resolution? These choices can be made from this notebook; they should not be silently inferred from function names.

## Acceptance scenarios for the next implementation

| Scenario | Required outcome | Status in this investigation |
|---|---|---|
| Fresh supported notebook, “show callers” | Discover → prepare → stage → run → inspect → save through normal UI/tools | **Gap:** no selected-target graph card; 1.5.5/2.0.0 mismatch |
| Wrong ABI/platform | Structured incompatibility and a supported next action; no false-ready state | **Source boundary established**, product recovery route untested |
| First Rust fixture | `helper` and `entry`; local helper target resolved, remote call explicitly unresolved | **Observed pass** on pinned native artifact |
| Two paths with repeated local IDs | Full-key joins preserve each file's calls without mixing nodes | **Observed pass:** portable SQL returns four correctly scoped call rows |
| NULL/empty/unsupported/invalid input | Preserve distinct outcome and diagnostic; subsequent query still works | **Observed engine pass; gap:** cached error loses original text |
| Cache replay and repeated lineage | Replay has zero missing keys/child refresh; two parent occurrences per input remain | **Observed pass:** 5 → 0 missing, 5 parents / 10 occurrences |
| Fresh connection | Cached facts readable; functions not described as loaded until readiness is re-established | **Observed native pass**; host restart auto-setup for graph untested |
| Source or extractor changes | New compatible keys or explicit invalidation/migration; old source is not silently cited | **Source contract only**, not exercised as a drift scenario here |
| Ambiguous temporal change | Preserve diagnostic and conservative change classification | **Observed pass:** delete/add plus ambiguity diagnostics |
| Export/read across the runtime boundary | Managed SQL can inspect portable typed facts without claiming the extension is loaded | **Observed pass:** native 2.0.0 export → managed 1.5.5 query |
| Result exceeds display budget | Complete typed rows are retrievable with identity and paging/completeness metadata | **Gap:** current SQL read surface exposes saved table text |
| Interrupted setup reply / target change / cancellation | Exact retry or actionable recovery; no stale readiness or duplicate work claim | **Existing generic contracts/source; graph end-to-end untested** |
| Repository history at scale | Admission/resource limits, equivalent workload and restart receipts hold for the supported recipe | **Unknown here**; prior experiments are references, not revalidated acceptance |

No new speed, accuracy, platform-support, full-repository resolution or production-readiness claim follows from the small fixtures. Existing history documentation explicitly keeps production admission/backpressure and distribution as further integration work.

## Solve: a bounded model of the missing journey

Catalog used: `workflow` / `bounded_trace`, with `workflow.initial_state_allowed`, `workflow.transition_allowed`, and `workflow.bounded_reachability`.

The seven transitions are a **caller-authored stage abstraction**, not seven tool calls or a time budget:

`Task → PackageKnown → CompatibleRuntime → Loaded → SchemaKnown → InputsStaged → ResultChecked → Saved`

- **Current selected managed path:** give the agent package knowledge through source lookup, then represent the known ABI mismatch as a transition to `Stalled`. With no runtime-switch edge encoded, synthesis returns **unsat / infeasible** for reaching Saved within this model.
- **Proposed contract:** supply each required stage transition. Synthesis returns **sat / solution**, including the complete state/event witness.
- The native experiment is an actual alternate process route. Its success is fully compatible with the narrow blocked managed-path model.
- Neither result proves product-wide impossibility, validates an implementation, determines UX quality, or makes these priorities mathematically optimal. The useful output is an explicit list of transitions a finished journey must supply and verify.

Persistence initially failed with `ArtifactCount: limit 512, attempted 513`. The models were rerun with `persist:false`; their results report `cached:true`. Exact executed requests, status, scope, solver version, unsat core/witness and the persistence error are saved in [solver-results.json](artifacts/2026-11-16-extension-agent-experience/solver-results.json). **No new solve_id was obtained**, and existing receipts were not removed.

In [10]:
import json
from pathlib import Path

solver_evidence = json.loads(Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-16-extension-agent-experience/solver-results.json').read_text())
for label in ['current_managed_path', 'proposed_contract']:
    result = solver_evidence[label]['result'][0]
    print(label, '=>', result['status'], '/', result['outcome'])
    print('scope:', result['evaluation_scope'])
    if result.get('unsat_core'): print('unsat core:', result['unsat_core'])
    for assignment in result.get('assignments', []):
        if '.states[' in assignment['field']: print(' ', assignment['field'], '=', assignment['value'])
    print()


current_managed_path => unsat / infeasible
scope: {'kind': 'bounded_trace', 'horizon': 7, 'reachability': [{'rule_id': 'workflow.bounded_reachability', 'binding_index': 2, 'effective_bound': 7}]}
unsat core: ['workflow_rule_1_workflow_transition_allowed', 'workflow_rule_2_workflow_bounded_reachability']

proposed_contract => sat / solution
scope: {'kind': 'bounded_trace', 'horizon': 7, 'reachability': [{'rule_id': 'workflow.bounded_reachability', 'binding_index': 2, 'effective_bound': 7}]}
  traces.journey.states[1] = PackageKnown
  traces.journey.states[2] = CompatibleRuntime
  traces.journey.states[3] = Loaded
  traces.journey.states[4] = SchemaKnown
  traces.journey.states[5] = InputsStaged
  traces.journey.states[6] = ResultChecked
  traces.journey.states[7] = Saved



## Reproduce, inspect and decide

Read the tables above, then inspect these receipts:

- [Native probe source](artifacts/2026-11-16-extension-agent-experience/native_probe.py) and [complete native result](artifacts/2026-11-16-extension-agent-experience/native-results.json): exact runtime/artifact identity, guidance metadata, fixture evidence, cache replay and fresh-connection checks.
- [MCP observations](artifacts/2026-11-16-extension-agent-experience/mcp-observations.json): selected target, refreshed registry, partial reference search and saved extension membership. All are as-of observations.
- [Host source evidence](artifacts/2026-11-16-extension-agent-experience/host-source-evidence.json): nine current source-file hashes and the explicit limit of source-based UI assessment.
- [Solver requests and results](artifacts/2026-11-16-extension-agent-experience/solver-results.json): finite model scope and exact returned envelopes.
- [Native API contract](../../crates/spur-graph-duckdb/README.md), [full history evaluation](../../crates/spur-graph-duckdb/examples/full-temporal-evaluation.md), and [bounded temporal finalization](../../crates/spur-graph-duckdb/examples/bounded-temporal-finalization.md): current source references; history experiments were not rerun here.

The Python probe cell uses the already-present pinned environment under `/private/tmp` and a prior local extension artifact. Those paths are machine-local prerequisites, not an onboarding promise. The script re-creates only this experiment's `journey.duckdb` and regenerates its typed exports. Persisted table outputs remain inspectable; rerunning native extraction elsewhere requires preparing the documented exact runtime/artifact first.

**Recommended next specification:** W1–W4 as one vertical task: “From a fresh supported notebook, a user or agent can produce an evidence-bearing single-file caller result, inspect unresolved calls, save typed facts, and reopen with truthful readiness.” Repository orchestration can then build on that checked contract.